# Face Landmark Preprocessing & Visualization

Notebook này dùng để nhìn rõ dữ liệu landmark mặt trước và sau tiền xử lý.

Mục tiêu:
- Kiểm tra `landmarks.csv` có bao nhiêu class và feature.
- Trực quan landmark raw trước preprocessing.
- Normalize landmark theo khuôn mặt để giảm lệ thuộc vị trí/khoảng cách camera.
- Tính EAR/MAR để xem `drowsy` và `yawning` có tách rõ không.
- Train/evaluate nhanh để nhìn confusion matrix.

Gợi ý đọc kết quả: `safe` nên là fallback/rule loại trừ, còn `drowsy` và `yawning` nên được kiểm tra bằng metric rõ ràng như EAR/MAR.

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## 1. Tìm và load dataset

Notebook này nằm trong `backend/driver_training/analysis`, còn CSV mặc định nằm ở `backend/driver_training/collect/data/landmarks.csv`.

In [ ]:
NOTEBOOK_DIR = Path.cwd()

candidate_paths = [
    NOTEBOOK_DIR / ".." / "collect" / "data" / "landmarks.csv",
    NOTEBOOK_DIR / "collect" / "data" / "landmarks.csv",
    Path("../collect/data/landmarks.csv"),
    Path("DiQuaMuaHaa/backend/driver_training/collect/data/landmarks.csv"),
]

csv_path = next((p.resolve() for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Không tìm thấy collect/data/landmarks.csv. Hãy chỉnh csv_path thủ công.")

OUT_DIR = (NOTEBOOK_DIR / "outputs").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path, OUT_DIR

In [ ]:
df = pd.read_csv(csv_path, header=None)
labels = df.iloc[:, 0].astype(str)
X_raw = df.iloc[:, 1:].astype("float32")

n_samples, n_features = X_raw.shape
n_landmarks = n_features // 3

print(f"CSV: {csv_path}")
print(f"Samples: {n_samples}")
print(f"Features: {n_features}")
print(f"Landmarks: {n_landmarks} (features / 3)")
print("Class counts:")
display(labels.value_counts())

if n_features % 3 != 0:
    warnings.warn("Feature count không chia hết cho 3, CSV có thể hỏng.")

if "safe" not in set(labels):
    warnings.warn("Dataset hiện không có class 'safe'. Nếu safe là fallback thì OK, nhưng đừng train model 3-class từ CSV này.")

## 2. Kiểm tra dữ liệu raw trước tiền xử lý

Ở bước này ta chưa normalize gì cả. Mục tiêu là bắt lỗi data: thiếu class, NaN, feature lệch length, range bất thường.

In [ ]:
raw_summary = pd.DataFrame({
    "missing_values": [int(X_raw.isna().sum().sum())],
    "min": [float(X_raw.min().min())],
    "max": [float(X_raw.max().max())],
    "mean": [float(X_raw.to_numpy().mean())],
    "std": [float(X_raw.to_numpy().std())],
})
display(raw_summary)

fig, ax = plt.subplots(figsize=(8, 4))
labels.value_counts().plot(kind="bar", ax=ax, color="#3b82f6")
ax.set_title("Class Distribution - Raw CSV")
ax.set_xlabel("Label")
ax.set_ylabel("Samples")
plt.tight_layout()
fig.savefig(OUT_DIR / "01_class_distribution_raw.png", dpi=160)
plt.show()

In [ ]:
def to_points(row):
    arr = np.asarray(row, dtype=np.float32)
    return arr.reshape(-1, 3)

def plot_face_points(points, title="", ax=None, color="#06b6d4"):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(points[:, 0], -points[:, 1], s=5, alpha=0.75, c=color)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("-y")
    return ax

unique_labels = sorted(labels.unique())
fig, axes = plt.subplots(1, len(unique_labels), figsize=(5 * len(unique_labels), 5))
if len(unique_labels) == 1:
    axes = [axes]

for ax, label in zip(axes, unique_labels):
    idx = labels[labels == label].index[0]
    pts = to_points(X_raw.iloc[idx].values)
    plot_face_points(pts, f"Raw sample: {label}", ax=ax)

plt.tight_layout()
fig.savefig(OUT_DIR / "02_raw_landmark_samples.png", dpi=160)
plt.show()

## 3. Tiền xử lý landmark

Preprocessing ở đây làm 3 việc:

1. Reshape vector `1434` thành `(478, 3)`.
2. Dịch toàn bộ landmark về tâm mặt, dùng midpoint 2 mắt làm gốc nếu có.
3. Scale theo khoảng cách hai mắt. Nếu không có mắt thì fallback theo bounding box.

Cách này giúp model bớt phụ thuộc khuôn mặt nằm bên trái/phải, gần/xa camera.

In [ ]:
# MediaPipe FaceMesh indices
L_EYE_OUTER = 33
R_EYE_OUTER = 263
MOUTH_UPPER_INNER = 13
MOUTH_LOWER_INNER = 14
MOUTH_LEFT_CORNER = 61
MOUTH_RIGHT_CORNER = 291

L_EYE = {"p1": 33, "p2": 160, "p3": 158, "p4": 133, "p5": 153, "p6": 144}
R_EYE = {"p1": 362, "p2": 385, "p3": 387, "p4": 263, "p5": 373, "p6": 380}

def dist(a, b):
    return float(np.linalg.norm(np.asarray(a[:2]) - np.asarray(b[:2])))

def normalize_face(points):
    points = np.asarray(points, dtype=np.float32).reshape(-1, 3)
    if len(points) > max(L_EYE_OUTER, R_EYE_OUTER):
        center = (points[L_EYE_OUTER] + points[R_EYE_OUTER]) / 2.0
        scale = dist(points[L_EYE_OUTER], points[R_EYE_OUTER])
    else:
        center = points.mean(axis=0)
        xy_span = points[:, :2].max(axis=0) - points[:, :2].min(axis=0)
        scale = float(max(xy_span.max(), 1e-6))
    if not np.isfinite(scale) or scale < 1e-6:
        scale = 1.0
    return ((points - center) / scale).astype(np.float32)

def compute_mar(points):
    points = np.asarray(points, dtype=np.float32).reshape(-1, 3)
    if len(points) <= max(MOUTH_UPPER_INNER, MOUTH_LOWER_INNER, MOUTH_LEFT_CORNER, MOUTH_RIGHT_CORNER):
        return np.nan
    vertical = dist(points[MOUTH_UPPER_INNER], points[MOUTH_LOWER_INNER])
    horizontal = dist(points[MOUTH_LEFT_CORNER], points[MOUTH_RIGHT_CORNER])
    return vertical / (horizontal + 1e-6)

def compute_ear(points, eye):
    points = np.asarray(points, dtype=np.float32).reshape(-1, 3)
    need = max(eye.values())
    if len(points) <= need:
        return np.nan
    p1, p2, p3, p4, p5, p6 = [points[eye[k]] for k in ["p1", "p2", "p3", "p4", "p5", "p6"]]
    return (dist(p2, p6) + dist(p3, p5)) / (2.0 * dist(p1, p4) + 1e-6)

def compute_both_ear(points):
    return np.nanmean([compute_ear(points, L_EYE), compute_ear(points, R_EYE)])

points_raw = np.stack([to_points(row) for row in X_raw.to_numpy()])
points_norm = np.stack([normalize_face(p) for p in points_raw])
X_norm = points_norm.reshape(n_samples, -1)

print(points_raw.shape, X_norm.shape)

## 4. Trực quan sau tiền xử lý

Nếu normalize tốt, các mặt sẽ gần cùng tâm/cùng scale hơn. Đây là bước rất quan trọng để phát hiện preprocessing sai.

In [ ]:
fig, axes = plt.subplots(2, len(unique_labels), figsize=(5 * len(unique_labels), 9))
if len(unique_labels) == 1:
    axes = np.array([[axes[0]], [axes[1]]])

for col, label in enumerate(unique_labels):
    idx = labels[labels == label].index[0]
    plot_face_points(points_raw[idx], f"Before: {label}", ax=axes[0, col], color="#f97316")
    plot_face_points(points_norm[idx], f"After normalize: {label}", ax=axes[1, col], color="#22c55e")

plt.tight_layout()
fig.savefig(OUT_DIR / "03_before_after_normalize_samples.png", dpi=160)
plt.show()

## 5. Metric sau tiền xử lý: MAR và EAR

- MAR cao: miệng mở to, thường dùng cho `yawning`.
- EAR thấp: mắt nhắm/lim dim, thường dùng cho `drowsy`.

Hai metric này là ratio nên ít phụ thuộc scale, nhưng vẫn nên tính sau khi kiểm tra landmark ổn.

In [ ]:
metrics = pd.DataFrame({
    "label": labels.values,
    "MAR": [compute_mar(p) for p in points_raw],
    "EAR": [compute_both_ear(p) for p in points_raw],
})

display(metrics.groupby("label")[["MAR", "EAR"]].describe().round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if sns:
    sns.boxplot(data=metrics, x="label", y="MAR", ax=axes[0])
    sns.stripplot(data=metrics, x="label", y="MAR", ax=axes[0], color="black", alpha=0.25, size=2)
    sns.boxplot(data=metrics, x="label", y="EAR", ax=axes[1])
    sns.stripplot(data=metrics, x="label", y="EAR", ax=axes[1], color="black", alpha=0.25, size=2)
else:
    metrics.boxplot(column="MAR", by="label", ax=axes[0])
    metrics.boxplot(column="EAR", by="label", ax=axes[1])
axes[0].set_title("MAR by Label")
axes[1].set_title("EAR by Label")
plt.suptitle("")
plt.tight_layout()
fig.savefig(OUT_DIR / "04_mar_ear_distribution.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for label in unique_labels:
    m = metrics[metrics["label"] == label]
    ax.scatter(m["EAR"], m["MAR"], s=18, alpha=0.7, label=label)
ax.set_xlabel("EAR (lower = eyes more closed)")
ax.set_ylabel("MAR (higher = mouth more open)")
ax.set_title("EAR vs MAR")
ax.legend()
plt.tight_layout()
fig.savefig(OUT_DIR / "05_ear_vs_mar.png", dpi=160)
plt.show()

## 6. PCA trước và sau preprocessing

PCA không phải bằng chứng cuối cùng, nhưng giúp nhìn nhanh class có đang tách cụm không. Nếu raw tách vì vị trí camera, còn normalized tách vì biểu cảm, đó là dấu hiệu tốt.

In [ ]:
def pca_2d(X):
    Xs = StandardScaler().fit_transform(X)
    return PCA(n_components=2, random_state=42).fit_transform(Xs)

raw_pca = pca_2d(X_raw.to_numpy())
norm_pca = pca_2d(X_norm)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label in unique_labels:
    mask = labels.values == label
    axes[0].scatter(raw_pca[mask, 0], raw_pca[mask, 1], s=18, alpha=0.7, label=label)
    axes[1].scatter(norm_pca[mask, 0], norm_pca[mask, 1], s=18, alpha=0.7, label=label)

axes[0].set_title("PCA - Raw Landmarks")
axes[1].set_title("PCA - Normalized Landmarks")
for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend()
plt.tight_layout()
fig.savefig(OUT_DIR / "06_pca_raw_vs_normalized.png", dpi=160)
plt.show()

## 7. Train/evaluate nhanh

Cell này không thay model production. Nó chỉ train nhanh trong notebook để xem data hiện tại có học được không và class nào đang bị nhầm.

Lưu ý: nếu CSV thiếu `safe`, model này không thể học `safe`. Khi đó `safe` nên được xử lý như fallback ở app/runtime.

In [ ]:
class_counts = labels.value_counts()
can_stratify = class_counts.min() >= 2 and len(class_counts) >= 2

X_train, X_val, y_train, y_val = train_test_split(
    X_norm,
    labels.values,
    test_size=0.2,
    random_state=42,
    stratify=labels.values if can_stratify else None,
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)),
])

model.fit(X_train, y_train)
pred = model.predict(X_val)

print(classification_report(y_val, pred, digits=3))

cm_labels = sorted(labels.unique())
cm = confusion_matrix(y_val, pred, labels=cm_labels)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=cm_labels).plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Quick Validation Confusion Matrix")
plt.tight_layout()
fig.savefig(OUT_DIR / "07_confusion_matrix_quick_model.png", dpi=160)
plt.show()